# 01 · Experimentacion — clasificador de vinos

> Este notebook es la **libreta de laboratorio** del proyecto. Aqui exploras los
> datos, pruebas modelos, decides hiperparametros. El codigo que funciona bien
> migra a `src/ejemplo_completo/`.

## Objetivo

- Entender el dataset `load_wine`.
- Comparar `LogisticRegression` vs. `RandomForestClassifier` como baseline.
- Validar que el pipeline modular de `src/` produce el mismo resultado.

## Que NO va en este notebook

- Codigo de produccion. Esto es exploracion.
- Logica que la app o la API vayan a usar. Esa vive en `src/`.

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete `ejemplo_completo` desde este notebook
SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

from ejemplo_completo.config import FEATURES_NUM, SEMILLA, TARGET
from ejemplo_completo.data import cargar, separar_X_y
from ejemplo_completo.features import construir_preprocesador
from ejemplo_completo.model import construir_pipeline

## 1. Mirar los datos

Nunca entrenes antes de haber visto el dataset. ¿Cuantas filas, cuantas clases,
distribucion balanceada o no?

In [ ]:
df = cargar()
print(f"forma: {df.shape}")
print(f"clases: {df[TARGET].value_counts().to_dict()}")
df.head()

In [ ]:
df.describe().round(2)

### Distribuciones por feature

Vale la pena mirar que escalas tienen las features (alcohol anda en 11-15, proline
anda en cientos a miles): sin estandarizar, los modelos lineales se van a sesgar.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(14, 7))
for ax, col in zip(axes.flatten(), FEATURES_NUM):
    ax.hist(df[col], bins=20, color="#4F69CC")
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=7)
# ocultar los axes sobrantes
for ax in axes.flatten()[len(FEATURES_NUM):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Baseline rapido con LogisticRegression

Antes de meter un bosque, mide con el modelo mas simple. Si logistic ya logra
accuracy alta, no necesitas uno mas complejo (ocurre seguido en Wine).

In [ ]:
X, y = separar_X_y(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEMILLA
)

logreg = Pipeline([
    ("preprocesador", construir_preprocesador()),
    ("clasificador", LogisticRegression(max_iter=1000, random_state=SEMILLA)),
])
logreg.fit(X_train, y_train)
acc_logreg = (logreg.predict(X_test) == y_test).mean()
print(f"accuracy LogisticRegression: {acc_logreg:.4f}")

## 3. RandomForest

El modelo que `src/model.py` usa en produccion. Debe dar al menos tan bueno
como el baseline.

In [ ]:
pipeline = construir_pipeline()
pipeline.fit(X_train, y_train)
acc_rf = (pipeline.predict(X_test) == y_test).mean()
print(f"accuracy RandomForest:       {acc_rf:.4f}")
print(f"delta vs. LogisticRegression: {acc_rf - acc_logreg:+.4f}")

## 4. Validacion cruzada

Una sola particion es ruidosa. 5-fold da una estimacion mas honesta.

In [ ]:
scores_logreg = cross_val_score(logreg, X, y, cv=5, scoring="accuracy")
scores_rf = cross_val_score(construir_pipeline(), X, y, cv=5, scoring="accuracy")
print(f"LogisticRegression  cv=5: {scores_logreg.mean():.4f} +/- {scores_logreg.std():.4f}")
print(f"RandomForest        cv=5: {scores_rf.mean():.4f} +/- {scores_rf.std():.4f}")

## 5. Matriz de confusion y feature importance

Accuracy no es toda la historia. Para RandomForest tambien podemos ver que
variables son las que mas contribuyen.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, pipeline.predict(X_test))
ConfusionMatrixDisplay(cm, display_labels=["cultivar_0", "cultivar_1", "cultivar_2"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title("Matriz de confusion")

clasificador = pipeline.named_steps["clasificador"]
importancias = pd.Series(clasificador.feature_importances_, index=FEATURES_NUM).sort_values()
importancias.plot.barh(ax=axes[1], color="#4F69CC")
axes[1].set_title("Importancia de features (RandomForest)")
axes[1].set_xlabel("importancia")

plt.tight_layout()
plt.show()

## 6. Decision y proximo paso

Lo aprendido en este notebook:

- Wine es un dataset separable: tanto LogisticRegression como RandomForest llegan a accuracy ~1.
- La estandarizacion importa (si quitas `StandardScaler`, logistic se cae).
- Las features mas discriminantes son `flavanoids`, `proline` y `color_intensity`.

**Que migra a `src/`:**

- La eleccion de modelo (`RandomForestClassifier`, 200 arboles, semilla 42) ya esta en `model.py`.
- El preprocesador (impute mediana + StandardScaler) ya esta en `features.py`.
- Nada mas de este notebook se usa en produccion. Las graficas y comparativas son documentacion.

**Proximo paso:**

```bash
make train          # entrena y guarda el modelo con la config final
make test           # verifica que el pipeline sigue entero
make serve-api      # expone POST /predict
```